# Lab 5 Workbook: Describe Each Class, Then Decide
## MPS311/439 Machine Learning

The same tasks as the [Lab 5 worksheet](https://ml.wxing.me/materials/lesson5/lab.html), with the code already in place: **nothing to copy**.

**What you do today.** The Python is given. What is left to you are the machine-learning decisions: the blanks `____` and the questions. That is the lab. A cell with a blank stops with an error until you fill it.

After the four Parts (0 to 3) come the after-class part and the **Advanced Study** (MPS439: investigation A is required). Only the folded Answers are not here: they stay on the [worksheet](https://ml.wxing.me/materials/lesson5/lab.html).

## What this lab is for

In the lecture we assumed how the data of each class is generated (one bell-shaped hill per class) and used Bayes' rule to ask which class more likely produced a point. Today you do the same on a new dataset, **wines**, and then go back to the **lumps** to fix QDA when its spread cannot be estimated reliably.

Most of the time you will **change one value, run the cell again and compare**. Before each step there is a question: **write down your own answer first**, then run.

## How to work

1. **File → Save a copy in Drive**, and work in the copy so your answers are saved.
2. Run the cells from top to bottom (**▶** or **Shift + Enter**); each cell needs the cells above it.
3. Where you see **Your answer**, double-click the cell and write your answer **before** you run the next code cell. A guess written down first is how you find out what you really expected.
4. Fill each `____`, run the cell, and compare with the **Success check** below it.
5. Stuck on a blank? Open the folded Hint. The folded Answers are on the [worksheet](https://ml.wxing.me/materials/lesson5/lab.html); if you fall behind, copy one and carry on.
6. The folded **Try with AI** boxes are for when you finish early, or after class.

# Part 0: Set up (4 minutes)



## 0.1 The setup cell

Run this cell without reading it closely. It is the same at the start of every lab from now on.

In [ ]:
import numpy as np                                        # numbers and arrays
import matplotlib.pyplot as plt                           # plots
import warnings                                           # controls Python's warning messages
from scipy.stats import norm                              # the bell curve (normal distribution)
from sklearn.datasets import load_wine, load_breast_cancer   # today's two datasets
from sklearn.model_selection import train_test_split, KFold, cross_val_score   # splitting and cross-validation
from sklearn.preprocessing import StandardScaler          # puts every column on one scale
from sklearn.inspection import DecisionBoundaryDisplay    # draws a boundary
from sklearn.linear_model import LogisticRegression       # the model of Lesson 4
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis   # today's models
warnings.filterwarnings("once")                           # show each warning only once

## The data block

Every lab uses **this same block** to load a dataset, split it and standardise it. To use another dataset you change only the two lines marked `← change`; everything after it keeps the same names. It appears twice today (1.0 and 3.0); only the two lines marked `← change` differ.

# Part 1: Three varieties of wine (12 minutes)

**The data.** 178 wines from one region of Italy, made from three grape varieties (0, 1, 2), with 13 chemical measurements each. Column 0 is the alcohol content (%).

## 1.0 Load the wines

In [ ]:
data = load_wine()                                        # ← change: which dataset
X = data.data                                             # the measurements (features)
y = data.target                                           # ← change if needed: the labels (here the variety: 0, 1, 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

**Success check:** `(133, 13) [44 53 36]`: 133 training wines, 13 measurements, and 44, 53 and 36 wines of the three varieties.

## 1.1 Describe each variety by its alcohol content

**Question first:** do you expect the three varieties to differ in alcohol by several percentage points, or by less than one or two?

**Your answer:** *(double-click to write)*

In [ ]:
for k in [0, 1, 2]:                                       # the three varieties
    alcohol_k = X_train[y_train == k, 0]                  # alcohol (column 0) of the training wines of variety k
    print(k, alcohol_k.mean(), alcohol_k.std(ddof=1))     # centre and spread of variety k

<details><summary><b>Success check</b> (open after you run the cell)</summary>

about `13.75 0.45`, `12.30 0.57` and `13.19 0.53`. The centres are less than 1.5 points apart and each spread is about half a point, so the three bell curves overlap a lot.

</details>

## 1.2 Bayes' rule for one wine

As in Note Section 3, the probability of variety $k$ is $P(Y=k)$ times the height of variety $k$'s bell curve, divided by the same product added over all varieties. $P(Y=k)$ is the fraction of training wines of variety $k$.

**Question first:** a wine has alcohol 13.32. Variety 2's bell curve is the highest there. Will variety 2 win clearly, or only just?

Fill the blank so that `prior` is $P(Y=k)$, then run. **Then change `x` to 12.5, 13.0 and 14.0, running the cell each time.**

**Your answer:** *(double-click to write)*

In [ ]:
x = 13.32                                                 # alcohol of one wine: then try 12.5, 13.0, 14.0
tops = np.zeros(3)                                        # P(Y=k) times height, one per variety
for k in [0, 1, 2]:                                       # the three varieties
    alcohol_k = X_train[y_train == k, 0]                  # alcohol of the training wines of variety k
    prior = (y_train == ____).mean()                      # P(Y=k): fraction of training wines of variety k
    height = norm.pdf(x, alcohol_k.mean(), alcohol_k.std(ddof=1))   # height of variety k's bell curve at x
    tops[k] = prior * height                              # P(Y=k) times height
print(tops / tops.sum())                                  # Bayes' rule: probability of each variety

<details><summary><b>Hint</b> (only if you are stuck)</summary>

inside the loop, the variety is called `k`.

</details>

<details><summary><b>Success check</b> (open after you run the cell)</summary>

at 13.32, about `[0.422 0.130 0.448]`. At 12.5, 13.0 and 14.0 the winners are varieties 1 (`0.738`), 2 (`0.482`) and 0 (`0.789`).

</details>

**Question:** at 13.32, variety 2's curve is about 1.3 times as high as variety 0's, yet it wins only by 0.448 to 0.422. Which number holds it back? And why does the bottom of Bayes' rule here add three terms, when the lecture added two?

**Your answer:** *(double-click to write)*

## 1.3 The library does the same

Run the cell. QDA fitted on the alcohol column alone describes each variety by one bell curve, exactly as you did.

In [ ]:
qda_one = QuadraticDiscriminantAnalysis()                 # one bell curve per variety
qda_one.fit(X_train[:, [0]], y_train)                     # alcohol only
print(qda_one.predict_proba([[13.32]]))                   # its probabilities for alcohol 13.32

**Success check:** `[[0.42217126 0.12971521 0.44811353]]`, the same as your numbers in 1.2.

<details><summary><b>Try with AI (if you finish early, or after class)</b></summary>

**Using AI: the four steps**

The **Try with AI** boxes use an AI assistant for the Python you do not know yet, while you stay in charge of the machine learning. Every box has the same four steps:

1. **You want:** say what you want in machine-learning words (for example, "a bell curve for each class on top of the histograms").
2. **Tell the AI:** the names of the variables it should use.
3. **Ask:** the request.
4. **Check:** compare the AI's result with a number or shape given on this page. If it does not match, the AI or your question is wrong. Find out which.

---

**You want:** the three bell curves on top of the data. (After class, run 1.0 again first: Part 3 replaces the wines with the lumps.)

**Tell the AI:** `X_train[:, 0]` is the alcohol of each training wine and `y_train` its variety (0, 1 or 2).

**Ask:** *"For each variety, draw a semi-transparent histogram of X_train[:, 0] with density=True, and on top the normal curve with that variety's mean and standard deviation (ddof=1), in the same colour. Add a dashed vertical line at 13.32."*

**Check:** the curves peak near 13.75, 12.30 and 13.19; at 13.32 their heights are about 0.56, 0.14 and 0.73.

</details>

# Part 2: One shared spread, or one per class? (15 minutes)



## 2.1 The two boundaries on the lecture's data

This is the hand-made data of Note Section 1: class 1 is tight (120 points), class 0 is spread out (240 points).

**Question first:** with one spread shared by both classes, will the boundary be straight or curved? And with one spread per class?

Fill in the model that uses one shared spread.

**Your answer:** *(double-click to write)*

In [ ]:
rng = np.random.default_rng(0)                            # the same random numbers as the lecture
X1 = rng.multivariate_normal([0, 0], [[0.35, 0.15], [0.15, 0.35]], 120)      # class 1: tight
X0 = rng.multivariate_normal([1.0, 0.6], [[2.2, -0.6], [-0.6, 1.6]], 240)    # class 0: spread out
X_two = np.vstack([X1, X0])                               # 360 points, 2 features
y_two = np.concatenate([np.ones(120), np.zeros(240)])     # their labels
model_shared = ____()                                     # one spread shared by both classes
model_shared.fit(X_two, y_two)
model_own = QuadraticDiscriminantAnalysis()               # one spread per class
model_own.fit(X_two, y_two)
print(model_shared.score(X_two, y_two))                   # accuracy, shared spread
print(model_own.score(X_two, y_two))                      # accuracy, one spread per class

<details><summary><b>Hint</b> (only if you are stuck)</summary>

both names were imported in Part 0. The lecture called the shared one LDA (Note Section 5).

</details>

<details><summary><b>Success check</b> (open after you run the cell)</summary>

`0.7194444444444444` and `0.8666666666666667`.

</details>

Now draw the two boundaries:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))         # two plots side by side
DecisionBoundaryDisplay.from_estimator(model_shared, X_two, response_method="predict", alpha=0.3, ax=axes[0])   # shade each predicted class
axes[0].scatter(X_two[:, 0], X_two[:, 1], c=y_two, s=10)  # the points, coloured by class
axes[0].set_title("one shared spread")
DecisionBoundaryDisplay.from_estimator(model_own, X_two, response_method="predict", alpha=0.3, ax=axes[1])
axes[1].scatter(X_two[:, 0], X_two[:, 1], c=y_two, s=10)
axes[1].set_title("one spread per class")
plt.show()

<details><summary><b>Success check</b> (open after you run the cell)</summary>

on the left the two regions meet along a straight line; on the right one region is an ellipse around the tight class.

</details>

**Question:** what does the left model assume about the two classes that is not true for this data?

**Your answer:** *(double-click to write)*

## 2.2 Three models on the wines

Back to the wines, all 13 measurements, the same 5 folds.

**Question first:** in the lecture, on the lumps, QDA came last (logistic regression 0.979, LDA 0.960, QDA 0.958). Will it come last on the wines too?

**Your answer:** *(double-click to write)*

In [ ]:
warnings.filterwarnings("ignore")                         # keep the output short
for model in [LogisticRegression(), LinearDiscriminantAnalysis(), QuadraticDiscriminantAnalysis()]:
    scores = cross_val_score(model, X_train_s, y_train, cv=kf)   # accuracy on each of the 5 folds
    print(model, scores.mean())                           # the model and its average accuracy

<details><summary><b>Success check</b> (open after you run the cell)</summary>

about `0.970`, `0.970` and `0.993`: on the wines QDA comes **first**.

</details>

Now use only two measurements. Run the cell with `[6, 9]` (flavanoids and colour intensity), then **choose two other columns** (any two of 0 to 12, for example `[0, 1]`) and run it again.

In [ ]:
cols = [6, 9]                                             # two columns: then choose your own
for model in [LogisticRegression(), LinearDiscriminantAnalysis(), QuadraticDiscriminantAnalysis()]:
    scores = cross_val_score(model, X_train_s[:, cols], y_train, cv=kf)   # only the chosen columns
    print(model, scores.mean())

<details><summary><b>Success check</b> (open after you run the cell)</summary>

with `[6, 9]`, about `0.902`, `0.895` and `0.932`. With `[0, 1]` the three are close, all about `0.77` to `0.79`.

</details>

**Questions:**

1. QDA is first on the wines but last on the lumps. Each wine variety has 36 to 53 training wines and 13 measurements (91 numbers per spread); each class of lumps has 159 or 267 lumps and 30 measurements (465 numbers per spread). Use these numbers to explain the difference.
2. Did the order of the three models stay the same for the columns you chose?

**Your answer:** *(double-click to write)*

# Part 3: When the spread cannot be estimated reliably (14 minutes)



## 3.0 Load the lumps

The data block again, with the two marked lines changed. The lumps are the data of Lessons 4 and 5; we want 1 for malignant, but scikit-learn uses 0, so we flip the labels.

In [ ]:
data = load_breast_cancer()                               # ← change: which dataset
X = data.data                                             # the measurements (features)
y = 1 - data.target                                       # ← change if needed: 1 = malignant, 0 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

**Success check:** `(426, 30) [267 159]`, the same split as Lab 4.

## 3.1 Why QDA warns

**Question first:** the perimeter of a lump is roughly 2π times its radius. Within the malignant lumps, how close to 1 do you expect the correlation between mean radius and mean perimeter to be?

Fill the column of the mean perimeter (column 2).

**Your answer:** *(double-click to write)*

In [ ]:
malignant = X_train[y_train == 1]                         # the malignant training lumps
corr = np.corrcoef(malignant[:, 0], malignant[:, ____])[0, 1]   # radius (column 0) against perimeter
print(corr)                                               # correlation within the malignant class
warnings.filterwarnings("always")                         # show the warning for this one fit
qda = QuadraticDiscriminantAnalysis()                     # one spread per class, no shrinking
qda.fit(X_train_s, y_train)                               # it warns here
warnings.filterwarnings("ignore")                         # hide warnings from here on

<details><summary><b>Hint</b> (only if you are stuck)</summary>

`malignant[:, 2]` is the mean perimeter.

</details>

<details><summary><b>Success check</b> (open after you run the cell)</summary>

a correlation of about `0.995`, then the warning *"The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity."* (On scikit-learn 1.8 or newer the fit stops with a `LinAlgError` instead: the same failure, reported as an error. Go on to 3.2.)

</details>

**Question:** the points lie almost on a line. What is the spread of a class across that line, and why does that trouble QDA, whose bell-shaped hill needs the inverse of the covariance matrix?

**Your answer:** *(double-click to write)*

<details><summary><b>Try with AI (if you finish early, or after class)</b></summary>

**You want:** to see the line the lumps lie on.

**Tell the AI:** `X_train` holds the training lumps (column 0 is the mean radius, column 2 the mean perimeter) and `y_train` the labels (1 = malignant).

**Ask:** *"Draw a scatter plot of column 2 against column 0 of X_train, with malignant lumps as filled blue dots and benign lumps as hollow grey circles."*

**Check:** the points of both classes lie close to one straight line, from about (7, 44) to about (28, 188).

</details>

## 3.2 Choose reg_param

Without shrinking, QDA scored 0.958 in the lecture. Run the cell, then **change the list to `[0.1, 0.2, 0.3]` and run it again**.

In [ ]:
for r in [0.0001, 0.001, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1]:   # values of reg_param: then try [0.1, 0.2, 0.3]
    qda = QuadraticDiscriminantAnalysis(reg_param=r)      # pull each spread towards a circle by r
    scores = cross_val_score(qda, X_train_s, y_train, cv=kf)   # accuracy on each fold
    print(r, round(scores.mean(), 3))                     # the value and its average accuracy

**Success check:** the full list gives

```
0.0001 0.962
0.001 0.955
0.01 0.958
0.02 0.962
0.05 0.969
0.1 0.967
0.2 0.965
0.3 0.965
0.5 0.967
1 0.927
```

and `[0.1, 0.2, 0.3]` gives `0.967`, `0.965`, `0.965`. The list leaves out 0: newer versions of scikit-learn stop with an error there, so in the assignment, too, start from a small positive value.

**Question:** which value is best in the full list, and is it at an end? With only `[0.1, 0.2, 0.3]`, where is the best value, and what should you do next?

**Your answer:** *(double-click to write)*

<details><summary><b>Try with AI (if you finish early, or after class)</b></summary>

**You want:** the curve of 3.2 as a picture.

**Tell the AI:** for each `r` in `[0.0001, 0.001, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1]`, `cross_val_score(QuadraticDiscriminantAnalysis(reg_param=r), X_train_s, y_train, cv=kf).mean()` is the accuracy.

**Ask:** *"Store the ten accuracies in a list and plot them against the values with markers, with a log scale on the horizontal axis and a dashed horizontal line at 0.958."*

**Check:** a peak of about `0.969` at 0.05 and a drop to about `0.927` at 1; compare Note Section 8.

</details>

## 3.3 The test set, once

All choosing is done. Fit with your chosen value and read the test set once.

In [ ]:
qda = QuadraticDiscriminantAnalysis(reg_param=____)       # the value chosen in 3.2
qda.fit(X_train_s, y_train)                               # train on all 426 training lumps
print(qda.score(X_test_s, y_test))                        # the test set, read once

**Success check:** about `0.958`. For comparison, on the same test lumps QDA without shrinking gets 0.937, LDA 0.951 and logistic regression (threshold 0.5) 0.986 (Note Section 8).

# After class: how today's lab helps with Assignment 2

Assignment 2 (handwritten digits, MNIST) asks for LDA and QDA in Section 2, and the MPS439 extension task C asks for a Gaussian classifier written by hand. **From today you can do the LDA and QDA rows of Table 2.**

| What you did today | Where it appears in Assignment 2 |
|---|---|
| The data block | Load the digits as the brief explains and give them the same names (`X_train`, `y_train`, ...); everything after that works unchanged |
| Part 2.2: LDA and QDA under one cross-validation | Section 2, Table 2: the LDA and QDA rows |
| Part 3.1: the "not full rank" warning, from a direction with almost no thickness | Section 2, the reminder "QDA with default settings fails". On the digits the flat directions come from pixels that are 0 in every image of a class: the same failure, only more extreme. scikit-learn before 1.8 gives the warning and a poor result, 1.8 or newer a `LinAlgError` |
| Part 3.2: try several `reg_param` values, starting above 0; the best must not sit at an end | Section 2: "The chosen value must not lie at either end of your search range" |
| Part 3.3: read the test set once | Section 4: `test` is used once, at the end |
| Advanced Study A below | MPS439 extension task C |

**Two differences in the assignment.** The brief divides the pixels by 255 instead of standardising them. And with 50,000 training images cross-validation is slow, so the assignment chooses `reg_param` on a separate **validation set**; the rule is the same as today: choose on data the final test never sees, and read the test set once at the end.

**Reflection** (after class, one or two sentences each):

1. Why do we multiply the height of each bell curve by $P(Y=k)$?
2. What does LDA assume that QDA does not, and what does that do to the boundary?
3. Why is QDA the best method on the wines but the worst on the lumps?
4. What does `reg_param = 1` do to every class's spread?

**Your answer:** *(double-click to write)*

# Advanced Study (after class, not part of the 50 minutes)

MPS311 students are encouraged to try one. **MPS439 students complete A**, the main task, and record it in the log at the end. Allow about 30 minutes for A. B and C are extras for everyone. These use the lumps of Part 3 (`X_train_s`, `X_test_s`, `y_train`, `y_test`).

## A. A Gaussian classifier in numpy

**Tasks**

1. Using only numpy, estimate for each class of lumps: the centre, the covariance matrix pulled towards a circle by `r = 0.5`, that is `(1 - r) * np.cov(...) + r * np.eye(30)`, and $P(Y=k)$, the fraction of training lumps in the class. (Advanced Study 4 of the Note uses `r = 0.1`; here you use `0.5`, so its numbers will not match yours.)
2. For every test lump in `X_test_s`, compute the log of $P(Y=k)$ times height for each class, and predict the class with the larger value. Advanced Study 4 of the Note shows the formula; you may drop the constant that is the same for both classes.
3. Compare your predictions with `QuadraticDiscriminantAnalysis(reg_param=0.5)`. Then take the benign training lumps, `X_train_s[y_train == 0]`, compute their spread `S` with `r = 0`, and print the smallest value of `np.linalg.eigvalsh(S)` (the spread in the flattest direction) and `np.linalg.slogdet(S)`. Explain what this does to `np.linalg.inv(S)` and to the height of the bell (Note Sections 4 and 8).

**Ask AI**

Ask: *"Why is the log of the Gaussian density equal to -0.5 times the squared distance measured by the class's own spread (the inverse covariance), minus half the log of the determinant, plus a constant?"* Check its steps against Section 4 of the Note.

<details><summary><b>Hint</b> (only if you are stuck)</summary>

`np.sum(diff @ np.linalg.inv(S) * diff, axis=1)` gives the squared distance of every row of `diff` at once, measured by the spread `S`. `np.linalg.slogdet(S)[1]` is the log of the determinant, and stays finite where `np.linalg.det` would underflow.

</details>

In [ ]:
# Your code here: add more cells with + Code if you need them

**Self-check** (run after your code, with your 0/1 predictions for the test lumps in `mine`)

In [ ]:
# Run this after your code for tasks 1 and 2, with your 0/1 predictions
# for the test lumps in mine
ref = QuadraticDiscriminantAnalysis(reg_param=0.5).fit(X_train_s, y_train)   # sklearn, the same shrinkage
print((mine != ref.predict(X_test_s)).sum())              # number of test lumps where the two disagree

The result should be `0`, and both have a test accuracy of about `0.958` (with the Note's `r = 0.1` it would be `0.951`). For the benign lumps with `r = 0`: the smallest spread is about `0.00003`, against a largest of about 5, and `slogdet` gives sign `1` and about `-98.7`. The inverse multiplies by about 1 / 0.00003, that is about 34,000, along the flattest direction, so a tiny change in a lump moves its distance, and its height, enormously.

## B. Why one shared spread gives a straight line

**Tasks**

1. Follow Advanced Study 2 of the Note and derive $\mathbf w = \Sigma^{-1}(\boldsymbol\mu_1-\boldsymbol\mu_0)$ and $b$ for LDA. Write the derivation in a text cell.
2. Compute the shared spread yourself: `((n1 - 1) * S1 + (n0 - 1) * S0) / (n1 + n0 - 2)`, where `S1`, `S0` are `np.cov` of each class of `X_train_s` and `n1`, `n0` the class sizes. Then compute `w` and `b`.
3. Compare `w` with `LinearDiscriminantAnalysis().fit(X_train_s, y_train).coef_[0]`, and `b` with its `intercept_`.

**Your derivation (task 1):** *(double-click to write)*

In [ ]:
# Your code here: add more cells with + Code if you need them

**Self-check:** the largest difference between the two sets of weights is below `1e-6`; the largest weight is about `19.2` and the intercept about `-2.45`.

## C. Two more ideas

Choose one.

1. **Naive Bayes.** `sklearn.naive_bayes.GaussianNB` assumes that the features in each class do not vary together: only the diagonal of each covariance matrix is kept. Compare its cross-validated accuracy (with `kf`) on `X_train_s` with LDA and QDA. How many numbers does it estimate?
2. **Different class probabilities.** A screening centre sees far fewer malignant lumps than this data set. Fit `QuadraticDiscriminantAnalysis(priors=[0.99, 0.01])` on the radius alone (`X_train[:, [0]]`) and print the probability of malignant for test lump 47 (radius 12.8), the opening lump of Lesson 4. Compare this with lowering the threshold in Lab 4.

In [ ]:
# Your code here: add more cells with + Code if you need them

**Self-check:** for naive Bayes, write the accuracy you found and the count of numbers; for different class probabilities, the probability should be far below `0.114`, the lecture's value.

## Advanced Study log

Double-click the cell below and fill it in.

> **Investigations done (A, B or C):**
>
> **What I found:**
>
> **How AI helped, and what I had to correct:**
>
> **Self-check result (passed or not):**

## 30-second feedback

Share anonymous feedback or post a question for this lab on the [course's Class page](https://ml.wxing.me/live/) (Give feedback, then choose Lesson 5 lab).